# Evaluación de SqueezeNet v1.1 (pesos de ImageNet) — 5 clases de residuos

Este notebook mide qué tan bien clasifica el modelo **SqueezeNet v1.1** ya entrenado, usando el set de **prueba** (`data/raw/waste_classification/test`). No entrena nada: solo carga el modelo y analiza sus predicciones.

**Qué se obtiene:**
1. Accuracy y loss globales.
2. Matriz de confusión y reporte de clasificación (precision, recall, F1).
3. Ejemplos visuales de imágenes mal clasificadas.
4. Accuracy por clase.

**Modelo evaluado:** `models/SqueezeNet/modelo_residuos_SqueezeNet_imagenet.keras` (generado por `entrenamiento_SqueezeNet.ipynb`).

> 📁 Este notebook vive en `notebooks/SqueezeNet/`, por eso las rutas parten de `Path("../..")` (dos niveles arriba = raíz del proyecto) y los modelos se leen/guardan en `models/SqueezeNet/`.

### Importación de librerías

Herramientas que se usan para evaluar el modelo:

* **`numpy`:** operaciones con arreglos (por ejemplo, elegir la clase con mayor probabilidad con `argmax`).
* **`matplotlib` y `seaborn`:** gráficas; `seaborn.heatmap` dibuja la matriz de confusión.
* **`tensorflow` / `load_model`:** carga el modelo `.keras` ya entrenado.
* **`ImageDataGenerator`:** lee las imágenes de prueba desde carpetas.
* **`classification_report` y `confusion_matrix` (scikit-learn):** calculan las métricas detalladas por clase.
* **`pathlib.Path`:** rutas que funcionan igual en Windows, Linux y Mac.

In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import load_model
from sklearn.metrics import classification_report, confusion_matrix
from pathlib import Path

2026-10-17 10:14:19.261107: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-10-17 10:14:19.298044: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-10-17 10:14:20.217954: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


### Configuración de rutas

Definimos de dónde se leen los datos y el modelo, y dónde se guardan las gráficas.

* **`BASE_DIR = Path("../..")`:** raíz del proyecto (dos niveles arriba de `notebooks/SqueezeNet/`).
* **`DATA_DIR`:** `data/raw/waste_classification/test`, el set de **prueba**. Son imágenes que el modelo nunca usó para ajustar sus pesos.
* **`MODEL_PATH`:** `models/SqueezeNet/modelo_residuos_SqueezeNet_imagenet.keras`, generado por `entrenamiento_SqueezeNet.ipynb`.
* **`FIGURES_DIR`:** `reports/figures/`, donde se guardan las figuras para el reporte.
* **`FIGURES_DIR.mkdir(...)`:** crea la carpeta de figuras si no existe.

In [2]:
# --- 1. CONFIGURACIÓN DE RUTAS (Compatible Windows/Linux) ---
BASE_DIR = Path("../..")
DATA_DIR = BASE_DIR / "data" / "raw" / "waste_classification" / "test"
MODELS_DIR = BASE_DIR / "models" / "SqueezeNet"
FIGURES_DIR = BASE_DIR / "reports" / "figures"

# Creamos la carpeta de figuras por si no existe
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Modelo generado por entrenamiento_SqueezeNet.ipynb
MODEL_PATH = MODELS_DIR / "modelo_residuos_SqueezeNet_imagenet.keras"

### Verificación de archivos y carga del modelo

Antes de hacer cualquier cálculo comprobamos que todo lo necesario existe, para obtener un error claro en vez de uno confuso más adelante.

1. Si no existe la carpeta de prueba, se detiene con `FileNotFoundError` indicando la ruta buscada.
2. Si no existe el modelo, se detiene pidiendo **ejecutar primero el notebook de entrenamiento**.
3. **Carga del modelo:** `load_model()` reconstruye la arquitectura (módulos *Fire* + cabecera) y los pesos desde el archivo `.keras`. Como se construyó íntegramente con capas nativas de Keras, se carga directamente sin requerir `custom_objects`.

In [3]:
# --- 2. VERIFICACIÓN Y CARGA ---
if not DATA_DIR.exists():
    raise FileNotFoundError(f"No se encontraron los datos en: {DATA_DIR}")
if not MODEL_PATH.exists():
    raise FileNotFoundError(f"No se encontró el modelo en: {MODEL_PATH}. ¡Ejecuta el entrenamiento primero!")

print("Rutas verificadas.")
print(f"Cargando modelo desde: {MODEL_PATH}")

model = load_model(MODEL_PATH)
print("✅ Modelo SqueezeNet v1.1 (ImageNet) cargado exitosamente.")

Rutas verificadas.
Cargando modelo desde: ../../models/SqueezeNet/modelo_residuos_SqueezeNet_imagenet.keras


InternalError: cudaSetDevice() on GPU:0 failed. Status: CUDA-capable device(s) is/are busy or unavailable